# Graph Learning Intuition

This notebook continues from the graph representation notebook and shows how graph structure becomes a learning signal.

Main ideas:
- node features are updated from local neighborhoods
- one message-passing step mixes information from adjacent nodes
- stacked message-passing steps widen the receptive field
- graph-level pooling turns node states into a molecule representation

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sns.set_theme(style="whitegrid")

In [ ]:
def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd.parent if cwd.name == 'notebooks' else cwd


PROJECT_ROOT = resolve_project_root()
DATA_DIR = PROJECT_ROOT / 'data'
bbbp = pd.read_csv(DATA_DIR / 'BBBP.csv')
print('BBBP shape:', bbbp.shape)

## 1. A Toy Molecular Graph

We start with a tiny graph whose node features encode simple atom-level properties.

In [ ]:
atom_labels = ['C1', 'C2', 'N3', 'O4', 'H5']
adjacency = np.array([
    [0, 1, 1, 0, 1],
    [1, 0, 0, 1, 0],
    [1, 0, 0, 1, 0],
    [0, 1, 1, 0, 0],
    [1, 0, 0, 0, 0],
], dtype=float)
node_features = pd.DataFrame({
    'atomic_number_proxy': [6, 6, 7, 8, 1],
    'is_hetero_atom': [0, 0, 1, 1, 0],
    'is_terminal': [0, 0, 0, 0, 1],
}, index=atom_labels)
display(node_features)

In [ ]:
degree = adjacency.sum(axis=1, keepdims=True)
normalized_adjacency = adjacency / np.clip(degree, a_min=1.0, a_max=None)
message_passing_step_1 = normalized_adjacency @ node_features.to_numpy(dtype=float)
updated_step_1 = 0.5 * node_features.to_numpy(dtype=float) + 0.5 * message_passing_step_1

updated_step_1_df = pd.DataFrame(updated_step_1, index=atom_labels, columns=node_features.columns)
display(updated_step_1_df.round(3))

After one message-passing step, each node representation becomes a mixture of its own features and its neighbors' features. This is the core idea behind graph neural networks.

In [ ]:
message_passing_step_2 = normalized_adjacency @ updated_step_1
updated_step_2 = 0.5 * updated_step_1 + 0.5 * message_passing_step_2
updated_step_2_df = pd.DataFrame(updated_step_2, index=atom_labels, columns=node_features.columns)
display(updated_step_2_df.round(3))

pooled_graph_embedding = updated_step_2.mean(axis=0)
print('Graph embedding:', np.round(pooled_graph_embedding, 3).tolist())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.heatmap(node_features, annot=True, cmap='Blues', ax=axes[0], cbar=False)
axes[0].set_title('Initial Node Features')
sns.heatmap(updated_step_1_df, annot=True, cmap='Greens', ax=axes[1], cbar=False)
axes[1].set_title('After 1 Message-Passing Step')
sns.heatmap(updated_step_2_df, annot=True, cmap='Oranges', ax=axes[2], cbar=False)
axes[2].set_title('After 2 Message-Passing Steps')
plt.tight_layout()
plt.show()

## 2. Neighborhood Aggregation on a BBBP Similarity Graph

The same idea works on a molecule-similarity graph: each molecule node can borrow signal from its nearest neighbors.

In [ ]:
sample = bbbp[['name', 'smiles', 'p_np']].head(20).copy()
vectorizer = CountVectorizer(analyzer='char', ngram_range=(2, 4), min_df=1)
feature_matrix = vectorizer.fit_transform(sample['smiles'])
similarity = cosine_similarity(feature_matrix)
similarity_graph = (similarity >= 0.45).astype(float)
np.fill_diagonal(similarity_graph, 0.0)
node_signal = sample[['p_np']].to_numpy(dtype=float)
similarity_degree = similarity_graph.sum(axis=1, keepdims=True)
normalized_similarity = similarity_graph / np.clip(similarity_degree, a_min=1.0, a_max=None)
neighbor_signal = normalized_similarity @ node_signal
comparison_df = pd.DataFrame({
    'molecule': sample['name'].fillna(sample.index.astype(str)),
    'original_label': node_signal.ravel(),
    'neighbor_average': neighbor_signal.ravel(),
    'degree': similarity_graph.sum(axis=1).astype(int),
})
display(comparison_df.head(10))

In [ ]:
plt.figure(figsize=(10, 4))
sns.scatterplot(data=comparison_df, x='degree', y='neighbor_average', hue='original_label', s=90)
plt.title('Neighborhood-Aggregated Signal on the BBBP Similarity Graph')
plt.xlabel('Graph degree')
plt.ylabel('Average neighbor label')
plt.show()

## 3. Message Passing as Matrix Operations

A common abstraction is:
- start with node states $H^{(0)}$
- propagate with a normalized adjacency-like matrix
- apply a learned transformation between propagation steps
- pool node states into a graph representation

A simplified update looks like $H^{(k+1)} = igma(at{A} H^{(k)} W^{(k)})$.

In [ ]:
weight_matrix = np.array([
    [0.2, -0.1],
    [0.5, 0.7],
    [-0.3, 0.6],
])
linear_projection = updated_step_2 @ weight_matrix
activated_projection = np.maximum(linear_projection, 0.0)
projection_df = pd.DataFrame(activated_projection, index=atom_labels, columns=['hidden_1', 'hidden_2'])
display(projection_df.round(3))

## Readiness for GNN Work

You are ready to move into graph neural networks when you can explain:
- what information a node receives after one and two propagation steps
- why pooling is needed for graph-level prediction
- why deeper message passing can both help and oversmooth
- how similarity graphs and atom-bond graphs support different learning problems